<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/week6_day2_dalychallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## Module d'attention multi-têtes (MultiHeadAttention)

Maintenant, nous allons étendre le concept d'attention à une seule tête pour implémenter l'attention multi-têtes. L'attention multi-têtes permet au modèle d'apprendre différentes représentations d'attention à partir de différentes "têtes" d'attention, ce qui peut capter des relations plus riches et diverses dans les données. Chaque tête d'attention fonctionne indépendamment, puis leurs sorties sont concaténées et projetées linéairement.

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        assert hidden_dim % num_heads == 0, "hidden_dim doit être divisible par num_heads"

        self.query_projection = nn.Linear(hidden_dim, hidden_dim)
        self.key_projection = nn.Linear(hidden_dim, hidden_dim)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim)
        self.output_projection = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # Linear projections for Q, K, V
        Q = self.query_projection(x)
        K = self.key_projection(x)
        V = self.value_projection(x)

        # Reshape and transpose for multi-head attention
        # (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, num_heads, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Calculate attention scores
        # (batch_size, num_heads, seq_len, head_dim) @ (batch_size, num_heads, head_dim, seq_len)
        # -> (batch_size, num_heads, seq_len, seq_len)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1))

        # Scale attention scores
        attention_scores = attention_scores / (self.head_dim ** 0.5)

        # Apply softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout

        # Multiply weights by values
        # (batch_size, num_heads, seq_len, seq_len) @ (batch_size, num_heads, seq_len, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        attention_output = torch.matmul(attention_weights, V)

        # Concatenate heads and apply final linear projection
        # (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        # -> (batch_size, seq_len, hidden_dim)
        attention_output = attention_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)

        # Apply output linear projection
        output = self.output_projection(attention_output)

        # No residual connection here yet, as it's typically added in the EncoderBlock
        return output, attention_weights


### Validation de `MultiHeadAttention` avec des tenseurs factices

Vérifions les formes d'entrée et de sortie du module `MultiHeadAttention`.

In [3]:
# Paramètres pour les tenseurs factices
batch_size = 2
seq_len = 10
hidden_dim = 64 # Doit être divisible par num_heads
num_heads = 8
dropout_rate = 0.1

# Création d'un tenseur d'entrée factice
dummy_input_mha = torch.randn(batch_size, seq_len, hidden_dim)
print(f"Forme de l'entrée factice pour MHA: {dummy_input_mha.shape}")

# Instanciation du module d'attention multi-têtes
multi_head_attention = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)

# Passage de l'entrée factice à travers le module
output_mha, attention_weights_mha = multi_head_attention(dummy_input_mha)

print(f"Forme de la sortie MHA: {output_mha.shape}")
print(f"Forme des poids d'attention MHA: {attention_weights_mha.shape}")

# Vérification des formes attendues
assert output_mha.shape == (batch_size, seq_len, hidden_dim)
assert attention_weights_mha.shape == (batch_size, num_heads, seq_len, seq_len)

print("Les formes des tenseurs pour MultiHeadAttention sont correctes!")


Forme de l'entrée factice pour MHA: torch.Size([2, 10, 64])
Forme de la sortie MHA: torch.Size([2, 10, 64])
Forme des poids d'attention MHA: torch.Size([2, 8, 10, 10])
Les formes des tenseurs pour MultiHeadAttention sont correctes!


## (Optionnel) Pile d'encodeurs personnalisée et boucle d'entraînement

Pour entraîner notre pile d'encodeurs personnalisée, nous avons besoin d'un ensemble de données. Nous allons utiliser un ensemble de données NLI (Natural Language Inference) et le préparer pour notre modèle.

### 1. Installation des dépendances

Nous aurons besoin de la bibliothèque `transformers` de Hugging Face pour la tokenisation et potentiellement pour charger un modèle de base pour la comparaison.

In [4]:
!pip install transformers datasets accelerate -qqq

### 2. Chargement de l'ensemble de données NLI

Nous allons charger l'ensemble de données SNLI (Stanford Natural Language Inference) via la bibliothèque `datasets`.

In [7]:
from datasets import load_dataset

# Correcting the dataset path to the full repository name
try:
    dataset = load_dataset('stanfordnlp/snli', split='train[:10000]')
except Exception as e:
    print(f"Error loading SNLI: {e}. Trying GLUE QNLI as fallback.")
    dataset = load_dataset('glue', 'qnli', split='train[:10000]')

print(f"Nombre d'exemples dans le jeu de données: {len(dataset)}")
print("Un exemple de l'ensemble de données:")
print(dataset[0])

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Nombre d'exemples dans le jeu de données: 10000
Un exemple de l'ensemble de données:
{'premise': 'A person on a horse jumps over a broken down airplane.', 'hypothesis': 'A person is training his horse for a competition.', 'label': 1}


### 3. Tokenisation de l'ensemble de données

Nous allons utiliser un tokenizer pré-entraîné (par exemple, DistilBERT) pour transformer nos phrases en identifiants de jetons numériques.

In [8]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    p_key = 'premise' if 'premise' in examples else 'question'
    h_key = 'hypothesis' if 'hypothesis' in examples else 'sentence'
    return tokenizer(examples[p_key], examples[h_key], truncation=True, padding='max_length', max_length=128)

# Map tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Standardize labels and columns
tokenized_dataset = tokenized_dataset.rename_column('label', 'labels')
remaining_cols = [col for col in ['premise', 'hypothesis', 'question', 'sentence'] if col in tokenized_dataset.column_names]
tokenized_dataset = tokenized_dataset.remove_columns(remaining_cols)

# Filter invalid labels
tokenized_dataset = tokenized_dataset.filter(lambda x: x['labels'] != -1)

print("Tokenization completed successfully.")
print(f"Columns: {tokenized_dataset.column_names}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenization completed successfully.
Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


## 4. Modèle de Classification Transformer

Nous allons maintenant définir un modèle de classification qui utilise notre pile d'encodeurs (`EncoderStack`) personnalisée. Ce modèle prend les identifiants de jetons, les passe par une couche d'incorporation (embedding), puis à travers la pile d'encodeurs, et utilise enfin la représentation du premier jeton pour la classification.

In [18]:
class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout_rate=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(hidden_dim, num_heads, dropout_rate)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),
            nn.ReLU(),
            nn.Linear(4 * hidden_dim, hidden_dim)
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        attn_output, weights = self.mha(x)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x, weights

class EncoderStack(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderBlock(hidden_dim, num_heads, dropout_rate) for _ in range(num_layers)])

    def forward(self, x):
        all_weights = []
        for layer in self.layers:
            x, weights = layer(x)
            all_weights.append(weights)
        return x, all_weights

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_heads, num_layers, num_classes, max_seq_len=128, dropout_rate=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Parameter(torch.zeros(1, max_seq_len, hidden_dim))
        self.encoder = EncoderStack(hidden_dim, num_heads, num_layers, dropout_rate)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_ids):
        seq_len = input_ids.size(1)
        embeddings = self.token_embedding(input_ids) + self.position_embedding[:, :seq_len, :]
        embeddings = self.dropout(embeddings)
        encoded, _ = self.encoder(embeddings)
        cls_representation = encoded[:, 0, :]
        logits = self.classifier(cls_representation)
        return logits

# Configuration améliorée pour de meilleurs résultats
VOCAB_SIZE = tokenizer.vocab_size
HIDDEN_DIM = 256 # Augmenté de 128 à 256
NUM_HEADS = 8    # Augmenté de 4 à 8
NUM_LAYERS = 4   # Augmenté de 2 à 4
NUM_CLASSES = 3

model = TransformerClassifier(VOCAB_SIZE, HIDDEN_DIM, NUM_HEADS, NUM_LAYERS, NUM_CLASSES)
print("Modèle réinitialisé avec une architecture plus profonde.")

Modèle réinitialisé avec une architecture plus profonde.


## 5. Préparation du DataLoader et Boucle d'Entraînement

Nous convertissons le jeu de données Hugging Face au format PyTorch et lançons une courte session d'entraînement pour valider le processus.

In [13]:
from torch.utils.data import DataLoader

# Format the dataset for PyTorch
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
train_dataloader = DataLoader(tokenized_dataset, batch_size=16, shuffle=True)

# Training parameters - Fixed the device check logic
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

# Training loop (1 batch for validation)
model.train()
for batch in train_dataloader:
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    print(f"Initial Loss: {loss.item():.4f}")
    break

print("Training step successful with the corrected device logic.")

Initial Loss: 1.0515
Training step successful with the corrected device logic.


```markdown
### 6. Boucle d'entraînement complète et évaluation

Nous allons maintenant entraîner le modèle sur plusieurs époques et calculer la précision globale.
```

In [ ]:
import time

# Ré-initialisation de l'optimiseur pour le nouveau modèle
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

NB_EPOQUES = 20 # Augmenté à 20 pour une meilleure convergence
print(f"Début de l'entraînement optimisé sur {device}...")

for epoch in range(NB_EPOQUES):
    model.train()
    total_loss = 0
    temps_debut = time.time()

    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    precision = calculer_precision(train_dataloader, model, device)
    temps_fin = time.time()

    print(f"Époque [{epoch+1}/{NB_EPOQUES}] - Perte: {total_loss/len(train_dataloader):.4f} - Précision: {precision:.2f}% - Temps: {temps_fin - temps_debut:.2f}s")

print("Entraînement optimisé terminé !")

Début de l'entraînement optimisé sur cuda...
Époque [1/20] - Perte: 1.1165 - Précision: 40.32% - Temps: 18.58s
Époque [2/20] - Perte: 1.0913 - Précision: 42.22% - Temps: 18.82s
Époque [3/20] - Perte: 1.0838 - Précision: 43.24% - Temps: 19.04s
Époque [4/20] - Perte: 1.0627 - Précision: 47.17% - Temps: 18.55s
Époque [5/20] - Perte: 1.0447 - Précision: 50.50% - Temps: 18.61s
Époque [6/20] - Perte: 1.0159 - Précision: 53.03% - Temps: 18.63s
Époque [7/20] - Perte: 0.9941 - Précision: 55.79% - Temps: 18.78s
Époque [8/20] - Perte: 0.9724 - Précision: 56.64% - Temps: 18.76s
Époque [9/20] - Perte: 0.9513 - Précision: 59.05% - Temps: 18.65s
Époque [10/20] - Perte: 0.9262 - Précision: 61.96% - Temps: 18.69s
Époque [11/20] - Perte: 0.9022 - Précision: 64.62% - Temps: 18.71s
Époque [12/20] - Perte: 0.8822 - Précision: 62.49% - Temps: 18.87s
Époque [13/20] - Perte: 0.8559 - Précision: 68.07% - Temps: 18.64s
Époque [14/20] - Perte: 0.8373 - Précision: 69.49% - Temps: 18.80s
Époque [15/20] - Perte: 0.

```markdown
### 7. Test d'inférence personnalisé

Testons le modèle sur une nouvelle paire de phrases pour observer sa prédiction.
```

In [17]:
def predire_nli(premisse, hypothese, model, tokenizer, device):
    model.eval()
    # Encodage de la paire de phrases avec les paramètres utilisés lors de l'entraînement
    encodage = tokenizer(premisse, hypothese, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    input_ids = encodage['input_ids'].to(device)

    with torch.no_grad():
        outputs = model(input_ids)
        _, prediction = torch.max(outputs, dim=1)

    # Définition des classes SNLI
    classes = ['entailment (implication)', 'neutral', 'contradiction']
    return classes[prediction.item()]

# Nouveau test avec le modèle plus entraîné
p = "Un homme dort sur un canapé vert."
h = "Un homme se repose."
resultat = predire_nli(p, h, model, tokenizer, device)

print(f"Prémisse: {p}")
print(f"Hypothèse: {h}")
print(f"Nouvelle prédiction après 10 époques: {resultat}")

Prémisse: Un homme dort sur un canapé vert.
Hypothèse: Un homme se repose.
Nouvelle prédiction après 10 époques: neutral
